# Personal Finance Analyser — SQL Analysis

## Overview
This notebook loads the cleaned transactions dataset into a SQLite database and runs SQL queries to answer key financial questions.

## Questions we'll answer
1. What is the total spent per category?
2. What are the monthly income vs expenses?
3. What are the top 10 biggest transactions?
4. Which month had the highest spending?
5. What is the monthly savings rate?
6. What are the average daily expenses per category?

In [1]:
# Import libraries
import pandas as pd
import sqlite3
import os

# Load the cleaned dataset
df = pd.read_csv('../data/transactions_cleaned.csv')

# Create a SQLite database in the data folder
conn = sqlite3.connect('../data/finance.db')

# Write the dataframe to a SQL table called 'transactions'
df.to_sql('transactions', conn, if_exists='replace', index=False)

print("Database created successfully")
print(f"Table 'transactions' loaded with {len(df)} rows")

Database created successfully
Table 'transactions' loaded with 575 rows


In [2]:
# Query 1 - Total spent per category
query1 = """
SELECT 
    category,
    COUNT(*) as num_transactions,
    ROUND(SUM(amount), 2) as total_amount
FROM transactions
WHERE type = 'expense'
GROUP BY category
ORDER BY total_amount DESC;
"""

result1 = pd.read_sql_query(query1, conn)
print("=== Total Spent Per Category ===")
print(result1.to_string(index=False))

=== Total Spent Per Category ===
     category  num_transactions  total_amount
      Savings                70      19051.91
      Housing                54      18776.80
     Clothing                68       5356.83
    Transport                67       5053.15
         Food                74       5034.20
Entertainment                73       2622.80
     Personal                83       2340.85
       Health                54       2173.23


In [3]:
# Query 2 - Monthly income vs expenses
query2 = """
SELECT 
    month,
    month_name,
    ROUND(SUM(CASE WHEN type = 'income' THEN amount ELSE 0 END), 2) as total_income,
    ROUND(SUM(CASE WHEN type = 'expense' THEN amount ELSE 0 END), 2) as total_expenses,
    ROUND(SUM(CASE WHEN type = 'income' THEN amount ELSE 0 END) - 
          SUM(CASE WHEN type = 'expense' THEN amount ELSE 0 END), 2) as net_position
FROM transactions
GROUP BY month, month_name
ORDER BY month;
"""

result2 = pd.read_sql_query(query2, conn)
print("=== Monthly Income vs Expenses ===")
print(result2.to_string(index=False))

=== Monthly Income vs Expenses ===
 month month_name  total_income  total_expenses  net_position
     1    January       3172.04         3390.29       -218.25
     2   February       2200.00         4369.40      -2169.40
     3      March       2200.00         6007.72      -3807.72
     4      April       2333.23         5113.26      -2780.03
     5        May       2650.26         6106.68      -3456.42
     6       June       2200.00         5624.67      -3424.67
     7       July       2256.17         5357.40      -3101.23
     8     August       2753.58         5137.69      -2384.11
     9  September       2552.72         4092.51      -1539.79
    10    October       2596.24         5868.76      -3272.52
    11   November       3117.28         4829.23      -1711.95
    12   December       2249.96         4512.16      -2262.20


In [4]:
# Query 3 - Top 10 biggest transactions
query3 = """
SELECT 
    date,
    description,
    category,
    amount,
    type
FROM transactions
ORDER BY amount DESC
LIMIT 10;
"""

result3 = pd.read_sql_query(query3, conn)
print("=== Top 10 Biggest Transactions ===")
print(result3.to_string(index=False))

=== Top 10 Biggest Transactions ===
      date    description category  amount   type
2024-01-25 Monthly Salary   Salary  2200.0 income
2024-02-25 Monthly Salary   Salary  2200.0 income
2024-03-25 Monthly Salary   Salary  2200.0 income
2024-04-25 Monthly Salary   Salary  2200.0 income
2024-05-25 Monthly Salary   Salary  2200.0 income
2024-06-25 Monthly Salary   Salary  2200.0 income
2024-07-25 Monthly Salary   Salary  2200.0 income
2024-08-25 Monthly Salary   Salary  2200.0 income
2024-09-25 Monthly Salary   Salary  2200.0 income
2024-10-25 Monthly Salary   Salary  2200.0 income


In [5]:
# Query 3b - Top 10 biggest expense transactions
query3b = """
SELECT 
    date,
    description,
    category,
    amount
FROM transactions
WHERE type = 'expense'
ORDER BY amount DESC
LIMIT 10;
"""

result3b = pd.read_sql_query(query3b, conn)
print("=== Top 10 Biggest Expense Transactions ===")
print(result3b.to_string(index=False))

=== Top 10 Biggest Expense Transactions ===
      date description category  amount
2024-11-01        Rent  Housing 1188.91
2024-04-16 Council Tax  Housing 1152.85
2024-06-10        Rent  Housing 1029.44
2024-05-12 Council Tax  Housing  995.76
2024-05-10 Council Tax  Housing  980.84
2024-08-28        Rent  Housing  936.95
2024-10-21 Council Tax  Housing  922.95
2024-03-24        Rent  Housing  914.20
2024-06-23 Council Tax  Housing  880.33
2024-07-15 Council Tax  Housing  805.87


In [6]:
# Query 4 - Month with highest spending
query4 = """
SELECT 
    month_name,
    ROUND(SUM(amount), 2) as total_spent,
    COUNT(*) as num_transactions
FROM transactions
WHERE type = 'expense'
GROUP BY month, month_name
ORDER BY total_spent DESC;
"""

result4 = pd.read_sql_query(query4, conn)
print("=== Monthly Spending Ranked Highest to Lowest ===")
print(result4.to_string(index=False))

=== Monthly Spending Ranked Highest to Lowest ===
month_name  total_spent  num_transactions
       May      6106.68                50
     March      6007.72                41
   October      5868.76                56
      June      5624.67                45
      July      5357.40                43
    August      5137.69                45
     April      5113.26                50
  November      4829.23                38
  December      4512.16                44
  February      4369.40                41
 September      4092.51                46
   January      3390.29                44


In [8]:
# Query 5 - Monthly savings rate
query5 = """
SELECT 
    month_name,
    month,
    ROUND(SUM(CASE WHEN type = 'income' THEN amount ELSE 0 END), 2) as income,
    ROUND(SUM(CASE WHEN type = 'expense' THEN amount ELSE 0 END), 2) as expenses,
    ROUND(SUM(CASE WHEN type = 'income' THEN amount ELSE 0 END) - 
          SUM(CASE WHEN type = 'expense' THEN amount ELSE 0 END), 2) as savings,
    ROUND(
        (SUM(CASE WHEN type = 'income' THEN amount ELSE 0 END) - 
         SUM(CASE WHEN type = 'expense' THEN amount ELSE 0 END)) / 
         SUM(CASE WHEN type = 'income' THEN amount ELSE 0 END) * 100
    , 2) as savings_rate_pct
FROM transactions
GROUP BY month, month_name
ORDER BY month;
"""

result5 = pd.read_sql_query(query5, conn)
print("=== Monthly Savings Rate ===")
print(result5.to_string(index=False))

=== Monthly Savings Rate ===
month_name  month  income  expenses  savings  savings_rate_pct
   January      1 3172.04   3390.29  -218.25             -6.88
  February      2 2200.00   4369.40 -2169.40            -98.61
     March      3 2200.00   6007.72 -3807.72           -173.08
     April      4 2333.23   5113.26 -2780.03           -119.15
       May      5 2650.26   6106.68 -3456.42           -130.42
      June      6 2200.00   5624.67 -3424.67           -155.67
      July      7 2256.17   5357.40 -3101.23           -137.46
    August      8 2753.58   5137.69 -2384.11            -86.58
 September      9 2552.72   4092.51 -1539.79            -60.32
   October     10 2596.24   5868.76 -3272.52           -126.05
  November     11 3117.28   4829.23 -1711.95            -54.92
  December     12 2249.96   4512.16 -2262.20           -100.54


In [9]:
# Query 6 - Average transaction amount per category
query6 = """
SELECT 
    category,
    COUNT(*) as num_transactions,
    ROUND(AVG(amount), 2) as avg_transaction,
    ROUND(MIN(amount), 2) as min_transaction,
    ROUND(MAX(amount), 2) as max_transaction
FROM transactions
WHERE type = 'expense'
GROUP BY category
ORDER BY avg_transaction DESC;
"""

result6 = pd.read_sql_query(query6, conn)
print("=== Average Transaction Amount Per Category ===")
print(result6.to_string(index=False))

=== Average Transaction Amount Per Category ===
     category  num_transactions  avg_transaction  min_transaction  max_transaction
      Housing                54           347.72            33.50          1188.91
      Savings                70           272.17           102.90           397.86
     Clothing                68            78.78            21.60           149.30
    Transport                67            75.42            13.47           147.17
         Food                74            68.03            15.28           119.25
       Health                54            40.25            11.14            79.81
Entertainment                73            35.93             8.43            59.97
     Personal                83            28.20             5.30            49.84


In [10]:
# Save all queries as .sql files
queries = {
    '01_total_per_category.sql': query1,
    '02_monthly_income_vs_expenses.sql': query2,
    '03_top_10_expenses.sql': query3b,
    '04_highest_spending_month.sql': query4,
    '05_monthly_savings_rate.sql': query5,
    '06_avg_transaction_per_category.sql': query6
}

for filename, query in queries.items():
    filepath = os.path.join('..', 'sql', filename)
    with open(filepath, 'w') as f:
        f.write(query.strip())
    print(f"Saved: {filename}")

print("\nAll queries saved to sql folder")

Saved: 01_total_per_category.sql
Saved: 02_monthly_income_vs_expenses.sql
Saved: 03_top_10_expenses.sql
Saved: 04_highest_spending_month.sql
Saved: 05_monthly_savings_rate.sql
Saved: 06_avg_transaction_per_category.sql

All queries saved to sql folder


In [11]:
# Close the database connection
conn.close()
print("Database connection closed")

Database connection closed
